# Filtering and Extraction


In [ ]:
from pathlib import Path
import pandas as pd
import re
import matplotlib.pyplot as plt

folder = Path("/cluster/project/reddy/katja/NGS_pipeline/data/P3481_LUCA-TCRDMF5/extracted")
summary_files = sorted(folder.glob("*.filtered.summary.tsv"))

# keep only the 19 sample sheets for overview (exclude ori library control)
sample_files = [f for f in summary_files if "orilib" not in f.name.lower()]

print("Found", len(summary_files), "summary files total")
print("Using", len(sample_files), "sample summary files (excluding orilib)")
sample_files[:3]


In [ ]:
def parse_summary_tsv(path: Path):
    text = path.read_text()
    sample = path.name.replace(".filtered.summary.tsv", "")

    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]

    # metric/value block
    idx_metric = next(i for i, ln in enumerate(lines) if ln.lower().startswith("metric"))

    next_header_idx = len(lines)
    for i in range(idx_metric + 1, len(lines)):
        if lines[i].startswith("#"):
            continue
        if re.match(r"^[A-Za-z_]+\t[A-Za-z_]+$", lines[i]):
            next_header_idx = i
            break

    metrics = []
    for ln in lines[idx_metric + 1:next_header_idx]:
        if ln.startswith("#"):
            continue
        parts = re.split(r"\s+", ln)
        if len(parts) == 2 and parts[1].isdigit():
            metrics.append((sample, parts[0], int(parts[1])))

    metrics_df = pd.DataFrame(metrics, columns=["sample", "metric", "value"])

    # position/count block (supports mutated_position_1based or fail_position_1based)
    idx_pos = None
    for i, ln in enumerate(lines):
        ll = ln.lower()
        if ll.startswith("mutated_position_1based") or ll.startswith("fail_position_1based"):
            idx_pos = i
            break

    pos = []
    if idx_pos is not None:
        for ln in lines[idx_pos + 1:]:
            if ln.startswith("#"):
                continue
            parts = re.split(r"\s+", ln)
            if len(parts) == 2 and parts[0].isdigit() and parts[1].isdigit():
                pos.append((sample, int(parts[0]), int(parts[1])))

    pos_df = pd.DataFrame(pos, columns=["sample", "position_1based", "count"])

    return metrics_df, pos_df


In [ ]:
all_metrics = []
all_pos = []

for f in sample_files:
    mdf, pdf = parse_summary_tsv(f)
    all_metrics.append(mdf)
    all_pos.append(pdf)

metrics_long = pd.concat(all_metrics, ignore_index=True)
pos_long = pd.concat(all_pos, ignore_index=True)

metrics_wide = (
    metrics_long
    .pivot_table(index="sample", columns="metric", values="value", aggfunc="sum")
    .fillna(0)
    .astype(int)
)

if "total_reads" in metrics_wide.columns and "pass" in metrics_wide.columns:
    metrics_wide["pass_rate"] = metrics_wide["pass"] / metrics_wide["total_reads"]
else:
    metrics_wide["pass_rate"] = 0.0

fail_cols = [c for c in metrics_wide.columns if c.startswith("fail_")]
metrics_wide["total_fail"] = metrics_wide[fail_cols].sum(axis=1) if fail_cols else 0

metrics_wide = metrics_wide.sort_values("pass_rate", ascending=False)
metrics_wide.head()


In [ ]:
# Overview figure for all 19 samples
df = metrics_wide.copy()

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True, gridspec_kw={"height_ratios": [2, 1]})

# top: pass vs fail (stacked)
axes[0].bar(df.index, df["pass"], label="pass", color="#2ca02c")
axes[0].bar(df.index, df["total_fail"], bottom=df["pass"], label="total_fail", color="#d62728")
axes[0].set_ylabel("Reads")
axes[0].set_title("Filtering summary overview (19 samples)")
axes[0].legend(loc="upper right")

# bottom: pass rate
axes[1].bar(df.index, df["pass_rate"], color="#1f77b4")
axes[1].set_ylabel("Pass rate")
axes[1].set_ylim(0, 1)
axes[1].axhline(df["pass_rate"].mean(), color="black", linestyle="--", linewidth=1, label=f"mean={df['pass_rate'].mean():.3f}")
axes[1].legend(loc="lower right")

plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


In [ ]:
fail_cols = [c for c in metrics_wide.columns if c.startswith("fail_")]
df = metrics_wide[fail_cols].copy()
df["total_fail"] = df.sum(axis=1)
df = df.sort_values("total_fail", ascending=False).drop(columns="total_fail")

plt.figure(figsize=(14, 6))
bottom = None
for col in df.columns:
    if bottom is None:
        plt.bar(df.index, df[col], label=col)
        bottom = df[col].values
    else:
        plt.bar(df.index, df[col], bottom=bottom, label=col)
        bottom = bottom + df[col].values

plt.xticks(rotation=90)
plt.ylabel("Reads")
plt.title("Failure breakdown per sample")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
pos_wide = (
    pos_long
    .pivot_table(index="sample", columns="position_1based", values="count", aggfunc="sum")
    .fillna(0)
)

order = metrics_wide.index
pos_wide = pos_wide.reindex(order)

plt.figure(figsize=(11, 8))
plt.imshow(pos_wide.values, aspect="auto")
plt.yticks(range(len(pos_wide.index)), pos_wide.index)
plt.xticks(range(len(pos_wide.columns)), pos_wide.columns)
plt.xlabel("Position (1-based)")
plt.ylabel("Sample")
plt.title("Mutated-position counts (heatmap)")
plt.colorbar(label="Count")
plt.tight_layout()
plt.show()


In [ ]:
report = metrics_wide.copy()
report.to_csv(folder / "all_samples.filtered.summary.report.csv")
report.sort_values("pass_rate", ascending=False).head(10)
